# STIR-Net V1 — 08 spatial-backbone isolated overfit

This notebook isolates the **spatial branch** on the same complete BlastoSPIM first-overfit scene used in notebooks 04–07.

The experiment intentionally removes:

- temporal graph / Trackastra reasoning;
- co-reasoning;
- queries;
- Hungarian matching;
- existence / count losses;
- native query-mask rendering;
- temporal Gaussian priors.

It asks one narrow question:

> **Can the spatial CNN memorize the foreground / center heatmap / boundary geometry of this one 3D scene?**

Two fresh models are trained independently:

1. **Current physical-aware backbone** — the repository `SpatialEncoder` + `SpatialDecoder` using `AxisFactorizedConv`.
2. **Ordinary 3×3×3 control** — same reduced channel widths, same encoder/decoder levels, same spacing-aware downsample schedule, but ordinary `Conv3d(3, padding=1)` residual blocks.

Both use the exact current STIR-Net dense losses and the same five spatial input channels.

Default: **30 optimizer steps per model**, evaluation every 5 steps.  
If needed, the final continuation block can extend either run without rebuilding the notebook.

No STIR-Net source files are modified or monkey-patched by this notebook.


In [ ]:
from pathlib import Path
import gc
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from learned.stirnet import RefinementCriterion
from learned.stirnet.debugging.acceptance.first_overfit import (
    _repo_root,
    _reduced_config,
    build_real_batch,
)
from learned.stirnet.model.blocks import DownsampleBlock, UpsampleBlock
from learned.stirnet.model.spacing import (
    AcquisitionEmbedding,
    choose_downsample_stride,
    propagate_spacing,
)
from learned.stirnet.model.spatial_encoder import SpatialEncoder
from learned.stirnet.model.spatial_decoder import SpatialDecoder
from learned.stirnet.model.heads import DenseAuxiliaryHeads
from learned.stirnet.model.types import SpatialPyramid

SEED = 40266

TRAIN_STEPS = 30
EVAL_EVERY = 5
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0

AMP_DTYPE = torch.float16
GRAD_SCALER_INITIAL_SCALE = 1024.0

# Stream dense targets from CPU in bounded chunks, matching the current criterion.
METRIC_CHUNK_VOXELS = 524_288

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "spatial_backbone_isolation"
    / "08_blastospim_first_overfit"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("This isolated spatial-backbone experiment requires CUDA.")

device = torch.device("cuda")

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))
print("CUDA       :", torch.version.cuda)
print("PyTorch    :", torch.__version__)
print("Steps/model:", TRAIN_STEPS)


## 1. Rebuild the exact previous BlastoSPIM scene

We reuse `build_real_batch(...)` from the first-overfit acceptance code. This keeps:

- the same all-cell ROI;
- the same target frame;
- the same 5 spatial channels;
- the same native voxel spacing;
- the same GT foreground, center-heatmap, and boundary targets.

Temporal tensors are built by the helper but are **never passed to either isolated model**.


In [ ]:
batch, sample = build_real_batch(DATA_DIR)

spatial_inputs_cpu = batch["spatial_inputs"].contiguous()
spacing_um_cpu = batch["spacing_um"].contiguous()
dref_um_cpu = batch["dref_um"].contiguous()
targets = batch["targets"]

assert spatial_inputs_cpu.shape[0] == 1
assert spatial_inputs_cpu.shape[1] == 5
assert len(targets) == 1

print("ROI shape          :", sample["roi_shape"])
print("Current cells      :", sample["current_count"])
print("GT cells           :", sample["target_count"])
print("Temporal tracklets :", sample["temporal_tracklets"], "(unused here)")
print("Spatial input      :", tuple(spatial_inputs_cpu.shape), spatial_inputs_cpu.dtype)
print("Spacing (um)       :", tuple(float(v) for v in spacing_um_cpu[0]))
print("dref (um)          :", float(dref_um_cpu[0]))

print("\nDense targets:")
for key in ("foreground", "center_heatmap", "boundary"):
    t = torch.as_tensor(targets[0][key])
    print(f"  {key:16s}", tuple(t.shape), t.dtype, float(t.min()), float(t.max()))


## 2. Reduced configuration used by the previous first-overfit experiment

This deliberately uses the same reduced spatial capacity as the earlier RTX-4050 experiment:

```text
channels = (4, 8, 16, 32)
blocks/level = 1
mask_dim = 8
```

That is important: this notebook tests the **actual reduced spatial backbone that participated in the failed same-sample overfit**, not the larger nominal V1 backbone.


In [ ]:
cfg = _reduced_config()

print("Spatial channels        :", cfg.spatial.channels)
print("Blocks per level        :", cfg.spatial.blocks_per_level)
print("Acquisition dim         :", cfg.spatial.acquisition_dim)
print("Anisotropy threshold    :", cfg.spatial.anisotropy_threshold)
print("Dense chunk voxels      :", cfg.losses.dense_chunk_voxels)
print("Activation checkpointing:", cfg.training.activation_checkpointing)
print("Checkpoint spatial      :", cfg.training.checkpoint_spatial)
print("Checkpoint losses       :", cfg.training.checkpoint_losses)


## 3. Model A — current physical-aware spatial backbone

This is repository code only:

```text
5-channel input
    ↓
AcquisitionEmbedding
    ↓
SpatialEncoder
    ↓
SpatialDecoder
    ↓
DenseAuxiliaryHeads
    ├── foreground
    ├── center heatmap
    └── boundary
```

No temporal stream and no co-reasoning are present.


In [ ]:
class PhysicalAwareSpatialOnly(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        checkpoint_spatial = (
            cfg.training.activation_checkpointing
            and cfg.training.checkpoint_spatial
        )
        self.cfg = cfg
        self.acquisition = AcquisitionEmbedding(cfg.spatial.acquisition_dim)
        self.encoder = SpatialEncoder(
            cfg.spatial,
            activation_checkpointing=checkpoint_spatial,
        )
        self.decoder = SpatialDecoder(
            cfg.spatial,
            activation_checkpointing=checkpoint_spatial,
        )
        self.dense_heads = DenseAuxiliaryHeads(cfg.spatial.channels[0])

    def forward(
        self,
        spatial_inputs,
        spacing_um,
        dref_um,
        spatial_padding_mask=None,
    ):
        acq = self.acquisition(spacing_um, dref_um)

        pyramid = self.encoder(
            spatial_inputs,
            spacing_um,
            acq,
            spatial_padding_mask,
        )

        # No CR1 / CR2. This experiment isolates encoder + decoder only.
        e3 = pyramid.features[3]
        e2 = self.decoder.decode_to_e2(
            e3,
            pyramid,
            acq,
        )
        # Skip the native mask projection entirely: this experiment isolates
        # the spatial backbone through the dense voxel-level heads only.
        d1 = self.decoder.stage_e1(
            e2,
            pyramid.features[1],
            acq,
        )
        d0 = self.decoder.stage_e0(
            d1,
            pyramid.features[0],
            acq,
        )

        dense = self.dense_heads(d0)

        return {
            "dense": dense,
            "pyramid": pyramid,
            "d0": d0,
        }


## 4. Model B — ordinary 3×3×3 control

This is a notebook-local control model. It intentionally preserves:

- the same four reduced channel widths;
- the same number of residual blocks per level;
- the same native input lattice;
- the same spacing-aware dynamic downsample schedule;
- the same trilinear decoder;
- the same dense heads.

The only major spatial operator change is:

```text
AxisFactorizedConv + spacing gates
        → ordinary Conv3d(3×3×3)
```

This is a **control**, not a proposed final STIR-Net architecture.


In [ ]:
def _groups(channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


class StandardResBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        max_groups: int = 8,
    ):
        super().__init__()
        self.pre = (
            nn.Conv3d(in_channels, out_channels, 1, bias=False)
            if in_channels != out_channels
            else nn.Identity()
        )
        self.norm1 = nn.GroupNorm(
            _groups(out_channels, max_groups),
            out_channels,
        )
        self.norm2 = nn.GroupNorm(
            _groups(out_channels, max_groups),
            out_channels,
        )
        self.conv1 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False,
        )
        self.conv2 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False,
        )

    def forward(self, x):
        residual = self.pre(x)
        x = self.conv1(F.silu(self.norm1(residual)))
        x = self.conv2(F.silu(self.norm2(x)))
        return x + residual


class StandardSpatialEncoder(nn.Module):
    def __init__(self, spatial_cfg, *, activation_checkpointing=True):
        super().__init__()
        self.cfg = spatial_cfg
        self.activation_checkpointing = activation_checkpointing

        ch = spatial_cfg.channels

        self.stem = nn.Conv3d(
            spatial_cfg.in_channels,
            ch[0],
            1,
            bias=False,
        )

        self.levels = nn.ModuleList(
            [
                nn.ModuleList(
                    [
                        StandardResBlock(
                            c,
                            c,
                            spatial_cfg.group_norm_max_groups,
                        )
                        for _ in range(spatial_cfg.blocks_per_level)
                    ]
                )
                for c in ch
            ]
        )

        combos = [
            (1, 1, 2),
            (1, 2, 1),
            (2, 1, 1),
            (1, 2, 2),
            (2, 1, 2),
            (2, 2, 1),
            (2, 2, 2),
        ]

        self.downs = nn.ModuleList()

        for i in range(len(ch) - 1):
            bank = nn.ModuleDict(
                {
                    "".join(map(str, stride)): DownsampleBlock(
                        ch[i],
                        ch[i + 1],
                        stride,
                    )
                    for stride in combos
                }
            )
            self.downs.append(bank)

    def forward(
        self,
        x,
        spacing_um,
        spatial_padding_mask=None,
    ):
        features = []
        spacings = []
        strides = []
        masks = []

        x = self.stem(x)
        current_spacing = spacing_um
        current_mask = spatial_padding_mask

        for level_idx, blocks in enumerate(self.levels):

            def run_level(level_input, level_blocks=blocks):
                result = level_input
                for block in level_blocks:
                    result = block(result)
                return result

            if self.activation_checkpointing and self.training:
                x = checkpoint(
                    run_level,
                    x,
                    use_reentrant=False,
                )
            else:
                x = run_level(x)

            features.append(x)
            spacings.append(current_spacing)

            if current_mask is not None:
                masks.append(current_mask)

            if level_idx < len(self.levels) - 1:
                stride = choose_downsample_stride(
                    current_spacing,
                    self.cfg.anisotropy_threshold,
                )
                strides.append(stride)

                key = "".join(map(str, stride))
                x = self.downs[level_idx][key](x)

                current_spacing = propagate_spacing(
                    current_spacing,
                    stride,
                )

                if current_mask is not None:
                    current_mask = F.max_pool3d(
                        current_mask.float().unsqueeze(1),
                        kernel_size=stride,
                        stride=stride,
                    ).squeeze(1).bool()

        return SpatialPyramid(
            features=features,
            spacings_um=spacings,
            strides=strides,
            padding_masks=(
                masks
                if spatial_padding_mask is not None
                else None
            ),
        )


class StandardDecoderStage(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        spatial_cfg,
        *,
        activation_checkpointing=True,
    ):
        super().__init__()
        self.activation_checkpointing = activation_checkpointing

        self.up = UpsampleBlock(
            in_channels,
            out_channels,
        )

        self.fuse = nn.Conv3d(
            out_channels + skip_channels,
            out_channels,
            1,
            bias=False,
        )

        self.blocks = nn.ModuleList(
            [
                StandardResBlock(
                    out_channels,
                    out_channels,
                    spatial_cfg.group_norm_max_groups,
                )
                for _ in range(spatial_cfg.blocks_per_level)
            ]
        )

    def _forward_impl(self, x, skip):
        x = self.up(
            x,
            skip.shape[-3:],
        )
        x = self.fuse(
            torch.cat([x, skip], dim=1)
        )
        for block in self.blocks:
            x = block(x)
        return x

    def forward(self, x, skip):
        if self.activation_checkpointing and self.training:
            return checkpoint(
                self._forward_impl,
                x,
                skip,
                use_reentrant=False,
            )
        return self._forward_impl(x, skip)


class StandardSpatialDecoder(nn.Module):
    def __init__(self, spatial_cfg, *, activation_checkpointing=True):
        super().__init__()
        ch = spatial_cfg.channels

        self.stage_e2 = StandardDecoderStage(
            ch[3],
            ch[2],
            ch[2],
            spatial_cfg,
            activation_checkpointing=activation_checkpointing,
        )
        self.stage_e1 = StandardDecoderStage(
            ch[2],
            ch[1],
            ch[1],
            spatial_cfg,
            activation_checkpointing=activation_checkpointing,
        )
        self.stage_e0 = StandardDecoderStage(
            ch[1],
            ch[0],
            ch[0],
            spatial_cfg,
            activation_checkpointing=activation_checkpointing,
        )

    def forward(self, pyramid):
        e3 = pyramid.features[3]

        e2 = self.stage_e2(
            e3,
            pyramid.features[2],
        )

        d1 = self.stage_e1(
            e2,
            pyramid.features[1],
        )

        d0 = self.stage_e0(
            d1,
            pyramid.features[0],
        )

        return d1, d0


class StandardConvSpatialOnly(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        checkpoint_spatial = (
            cfg.training.activation_checkpointing
            and cfg.training.checkpoint_spatial
        )

        self.encoder = StandardSpatialEncoder(
            cfg.spatial,
            activation_checkpointing=checkpoint_spatial,
        )

        self.decoder = StandardSpatialDecoder(
            cfg.spatial,
            activation_checkpointing=checkpoint_spatial,
        )

        self.dense_heads = DenseAuxiliaryHeads(
            cfg.spatial.channels[0]
        )

    def forward(
        self,
        spatial_inputs,
        spacing_um,
        dref_um,
        spatial_padding_mask=None,
    ):
        del dref_um  # control CNN does not use acquisition conditioning

        pyramid = self.encoder(
            spatial_inputs,
            spacing_um,
            spatial_padding_mask,
        )

        d1, d0 = self.decoder(pyramid)
        dense = self.dense_heads(d0)

        return {
            "dense": dense,
            "pyramid": pyramid,
            "d0": d0,
        }


## 5. Parameter-count sanity check

The ordinary 3×3×3 control is expected to have more convolution parameters than the axis-factorized model. This experiment is therefore **not** a capacity-matched benchmark.

Its role is diagnostic:

> if the current backbone cannot memorize one scene but the ordinary control can, the current spatial operator becomes a strong suspect.


In [ ]:
def parameter_count(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


torch.manual_seed(SEED)
physical_probe = PhysicalAwareSpatialOnly(cfg)

torch.manual_seed(SEED)
standard_probe = StandardConvSpatialOnly(cfg)

print(
    "Physical-aware parameters:",
    f"{parameter_count(physical_probe):,}",
)
print(
    "Ordinary 3x3x3 parameters:",
    f"{parameter_count(standard_probe):,}",
)

del physical_probe, standard_probe
gc.collect()
torch.cuda.empty_cache()


## 6. Reuse the exact current dense losses

Rather than inventing a new objective, we call the current `RefinementCriterion._dense_losses(...)` implementation directly.

For this isolated experiment:

```text
L_spatial =
    0.50 × foreground
  + 1.00 × center_heatmap
  + 0.50 × boundary
```

These are the same dense auxiliary terms used inside full STIR-Net. Query-specific losses are absent.


In [ ]:
criterion = RefinementCriterion(
    cfg.losses,
    cfg.queries,
    cfg.training,
).to(device)


def spatial_dense_loss(dense_outputs):
    foreground, center_heatmap, boundary = criterion._dense_losses(
        dense_outputs,
        targets,
    )

    total = (
        cfg.losses.foreground * foreground
        + cfg.losses.center_heatmap * center_heatmap
        + cfg.losses.boundary * boundary
    )

    return {
        "loss": total,
        "foreground": foreground,
        "center_heatmap": center_heatmap,
        "boundary": boundary,
    }


print("Dense-only loss weights:")
print("  foreground    :", cfg.losses.foreground)
print("  center_heatmap:", cfg.losses.center_heatmap)
print("  boundary      :", cfg.losses.boundary)


## 7. Bounded evaluation metrics

The primary decision metrics are:

- foreground hard Dice;
- boundary hard Dice;
- mean foreground probability inside vs outside GT;
- mean boundary probability on vs off GT.

We also record center-heatmap MSE and probability statistics.

Targets remain CPU-backed and are streamed to GPU in chunks.


In [ ]:
@torch.no_grad()
def _stream_binary_metrics(
    logits,
    target_cpu,
    *,
    threshold=0.5,
    chunk_voxels=METRIC_CHUNK_VOXELS,
):
    flat_logits = logits.detach().float().reshape(-1)
    flat_target = torch.as_tensor(target_cpu).reshape(-1)

    intersection = 0
    predicted_count = 0
    target_count = 0

    positive_prob_sum = 0.0
    positive_count = 0

    negative_prob_sum = 0.0
    negative_count = 0

    for start in range(0, flat_logits.numel(), chunk_voxels):
        end = min(
            start + chunk_voxels,
            flat_logits.numel(),
        )

        probability = torch.sigmoid(
            flat_logits[start:end]
        )

        target = flat_target[start:end].to(
            device=logits.device,
            dtype=torch.bool,
            non_blocking=True,
        )

        predicted = probability > threshold

        intersection += int(
            (predicted & target).sum().cpu()
        )
        predicted_count += int(
            predicted.sum().cpu()
        )
        target_count += int(
            target.sum().cpu()
        )

        if target.any():
            positive_prob_sum += float(
                probability[target].sum().cpu()
            )
            positive_count += int(
                target.sum().cpu()
            )

        negative = ~target
        if negative.any():
            negative_prob_sum += float(
                probability[negative].sum().cpu()
            )
            negative_count += int(
                negative.sum().cpu()
            )

    dice = (
        2.0 * intersection
        / max(
            predicted_count + target_count,
            1,
        )
    )

    return {
        "dice": float(dice),
        "predicted_voxels": int(predicted_count),
        "target_voxels": int(target_count),
        "mean_prob_positive": (
            positive_prob_sum
            / max(positive_count, 1)
        ),
        "mean_prob_negative": (
            negative_prob_sum
            / max(negative_count, 1)
        ),
    }


@torch.no_grad()
def _stream_regression_metrics(
    logits,
    target_cpu,
    *,
    chunk_voxels=METRIC_CHUNK_VOXELS,
):
    flat_logits = logits.detach().float().reshape(-1)
    flat_target = torch.as_tensor(target_cpu).reshape(-1)

    squared_error = 0.0
    count = 0

    high_target_prob_sum = 0.0
    high_target_count = 0

    low_target_prob_sum = 0.0
    low_target_count = 0

    for start in range(0, flat_logits.numel(), chunk_voxels):
        end = min(
            start + chunk_voxels,
            flat_logits.numel(),
        )

        probability = torch.sigmoid(
            flat_logits[start:end]
        )

        target = flat_target[start:end].to(
            device=logits.device,
            dtype=torch.float32,
            non_blocking=True,
        )

        squared_error += float(
            (probability - target)
            .square()
            .sum()
            .cpu()
        )
        count += end - start

        high = target > 0.5
        low = target < 0.05

        if high.any():
            high_target_prob_sum += float(
                probability[high].sum().cpu()
            )
            high_target_count += int(
                high.sum().cpu()
            )

        if low.any():
            low_target_prob_sum += float(
                probability[low].sum().cpu()
            )
            low_target_count += int(
                low.sum().cpu()
            )

    return {
        "mse": squared_error / max(count, 1),
        "mean_prob_target_gt_0.5": (
            high_target_prob_sum
            / max(high_target_count, 1)
        ),
        "mean_prob_target_lt_0.05": (
            low_target_prob_sum
            / max(low_target_count, 1)
        ),
    }


@torch.no_grad()
def evaluate_dense_outputs(dense_outputs):
    foreground = _stream_binary_metrics(
        dense_outputs["foreground_logits"][0, 0],
        targets[0]["foreground"],
    )

    boundary = _stream_binary_metrics(
        dense_outputs["boundary_logits"][0, 0],
        targets[0]["boundary"] > 0.5,
    )

    center = _stream_regression_metrics(
        dense_outputs["center_heatmap_logits"][0, 0],
        targets[0]["center_heatmap"],
    )

    return {
        "foreground_dice": foreground["dice"],
        "foreground_prob_inside": foreground["mean_prob_positive"],
        "foreground_prob_outside": foreground["mean_prob_negative"],
        "boundary_dice": boundary["dice"],
        "boundary_prob_on": boundary["mean_prob_positive"],
        "boundary_prob_off": boundary["mean_prob_negative"],
        "center_mse": center["mse"],
        "center_prob_high_target": center["mean_prob_target_gt_0.5"],
        "center_prob_low_target": center["mean_prob_target_lt_0.05"],
    }


## 8. Training helper

Each model is trained from a fresh initialization on the **same complete scene**.

Important details:

- batch size = 1;
- native scene is not cropped into individual-cell patches;
- FP16 autocast + GradScaler;
- spatial activation checkpointing remains enabled;
- GT dense maps stay on CPU;
- no optimizer state is shared between models.


In [ ]:
def _make_device_inputs():
    return {
        "spatial_inputs": spatial_inputs_cpu.to(
            device=device,
            dtype=AMP_DTYPE,
            non_blocking=True,
        ),
        "spacing_um": spacing_um_cpu.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        ),
        "dref_um": dref_um_cpu.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        ),
    }


@torch.no_grad()
def evaluate_model(model):
    model.eval()

    inputs = _make_device_inputs()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = model(**inputs)
        losses = spatial_dense_loss(
            outputs["dense"]
        )

    metrics = evaluate_dense_outputs(
        outputs["dense"]
    )

    row = {
        key: float(value.detach().float().cpu())
        for key, value in losses.items()
    }
    row.update(metrics)

    del outputs, inputs
    return row


def _tree_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {
            key: _tree_to_cpu(item)
            for key, item in value.items()
        }
    if isinstance(value, list):
        return [
            _tree_to_cpu(item)
            for item in value
        ]
    if isinstance(value, tuple):
        return tuple(
            _tree_to_cpu(item)
            for item in value
        )
    return value


def train_spatial_model(
    model_factory,
    *,
    name,
    steps=TRAIN_STEPS,
    eval_every=EVAL_EVERY,
    seed=SEED,
    initial_state=None,
    initial_history=None,
):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = model_factory().to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
        init_scale=GRAD_SCALER_INITIAL_SCALE,
    )

    if initial_state is not None:
        model.load_state_dict(
            initial_state["model"],
            strict=True,
        )
        optimizer.load_state_dict(
            initial_state["optimizer"]
        )
        scaler.load_state_dict(
            initial_state["scaler"]
        )

    history = (
        list(initial_history)
        if initial_history is not None
        else []
    )

    start_step = (
        int(history[-1]["step"])
        if history
        else 0
    )

    baseline = evaluate_model(model)
    if not history:
        history.append(
            {
                "step": 0,
                "phase": "eval",
                **baseline,
            }
        )

    print(f"\n{name}")
    print("-" * len(name))
    print(
        "baseline:",
        f"loss={baseline['loss']:.6f}",
        f"fg_dice={baseline['foreground_dice']:.4f}",
        f"boundary_dice={baseline['boundary_dice']:.4f}",
    )

    for local_step in range(1, steps + 1):
        global_step = start_step + local_step

        model.train()
        optimizer.zero_grad(set_to_none=True)

        inputs = _make_device_inputs()

        step_start = time.perf_counter()

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            outputs = model(**inputs)
            losses = spatial_dense_loss(
                outputs["dense"]
            )
            loss = losses["loss"]

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM,
        )

        scaler.step(optimizer)
        scaler.update()

        torch.cuda.synchronize()
        step_seconds = time.perf_counter() - step_start

        train_row = {
            "step": global_step,
            "phase": "train",
            "loss": float(
                loss.detach().float().cpu()
            ),
            "foreground": float(
                losses["foreground"]
                .detach()
                .float()
                .cpu()
            ),
            "center_heatmap": float(
                losses["center_heatmap"]
                .detach()
                .float()
                .cpu()
            ),
            "boundary": float(
                losses["boundary"]
                .detach()
                .float()
                .cpu()
            ),
            "grad_norm": float(
                torch.as_tensor(grad_norm)
                .detach()
                .float()
                .cpu()
            ),
            "scale": float(scaler.get_scale()),
            "seconds": float(step_seconds),
            "peak_cuda_gib": (
                torch.cuda.max_memory_allocated()
                / 1024**3
            ),
        }

        if (
            global_step % eval_every == 0
            or local_step == steps
        ):
            evaluation = evaluate_model(model)
            history.append(
                {
                    "step": global_step,
                    "phase": "eval",
                    **evaluation,
                    "grad_norm": train_row["grad_norm"],
                    "scale": train_row["scale"],
                    "seconds": train_row["seconds"],
                    "peak_cuda_gib": train_row["peak_cuda_gib"],
                }
            )

            print(
                f"step {global_step:03d} | "
                f"loss={evaluation['loss']:.6f} | "
                f"fg={evaluation['foreground_dice']:.4f} | "
                f"boundary={evaluation['boundary_dice']:.4f} | "
                f"fg in/out="
                f"{evaluation['foreground_prob_inside']:.3f}/"
                f"{evaluation['foreground_prob_outside']:.3f} | "
                f"{step_seconds:.1f}s | "
                f"peak={train_row['peak_cuda_gib']:.2f} GiB"
            )

        del outputs, inputs, losses, loss

    state = {
        "model": {
            key: value.detach().cpu()
            for key, value in model.state_dict().items()
        },
        "optimizer": _tree_to_cpu(
            optimizer.state_dict()
        ),
        "scaler": scaler.state_dict(),
    }

    final_eval = evaluate_model(model)

    del optimizer, scaler
    model.cpu()
    del model

    gc.collect()
    torch.cuda.empty_cache()

    return state, history, final_eval


## 9. Train Model A — current physical-aware backbone

Run this first. If it can memorize the scene, its foreground/boundary metrics should move decisively in the correct direction.


In [ ]:
physical_state, physical_history, physical_final = train_spatial_model(
    lambda: PhysicalAwareSpatialOnly(cfg),
    name="A — current physical-aware backbone",
)

physical_history_df = pd.DataFrame(physical_history)
physical_history_df


## 10. Train Model B — ordinary 3×3×3 control

The first model has been moved back to CPU and CUDA cache cleared before this run.


In [ ]:
standard_state, standard_history, standard_final = train_spatial_model(
    lambda: StandardConvSpatialOnly(cfg),
    name="B — ordinary 3x3x3 control",
)

standard_history_df = pd.DataFrame(standard_history)
standard_history_df


## 11. Direct comparison

The central diagnostic is not simply which model has lower loss. We want to know whether each model actually learns spatial separation:

```text
foreground probability inside GT  > outside GT
boundary probability on GT edge   > off GT edge
foreground Dice                    rises strongly
boundary Dice                      rises strongly
```


In [ ]:
comparison = pd.DataFrame(
    [
        {
            "model": "physical_aware",
            **physical_final,
        },
        {
            "model": "ordinary_3x3x3",
            **standard_final,
        },
    ]
)

comparison[
    [
        "model",
        "loss",
        "foreground_dice",
        "foreground_prob_inside",
        "foreground_prob_outside",
        "boundary_dice",
        "boundary_prob_on",
        "boundary_prob_off",
        "center_mse",
        "center_prob_high_target",
        "center_prob_low_target",
    ]
]


## 12. Learning curves


In [ ]:
def eval_rows(history):
    df = pd.DataFrame(history)
    return df[df["phase"] == "eval"].copy()


physical_eval = eval_rows(physical_history)
standard_eval = eval_rows(standard_history)

plt.figure(figsize=(8, 5))
plt.plot(
    physical_eval["step"],
    physical_eval["loss"],
    marker="o",
    label="physical-aware",
)
plt.plot(
    standard_eval["step"],
    standard_eval["loss"],
    marker="o",
    label="ordinary 3x3x3",
)
plt.xlabel("Optimizer step")
plt.ylabel("Dense-only loss")
plt.title("Spatial-only overfit loss")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    physical_eval["step"],
    physical_eval["foreground_dice"],
    marker="o",
    label="physical-aware",
)
plt.plot(
    standard_eval["step"],
    standard_eval["foreground_dice"],
    marker="o",
    label="ordinary 3x3x3",
)
plt.xlabel("Optimizer step")
plt.ylabel("Foreground Dice")
plt.title("Foreground memorization")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    physical_eval["step"],
    physical_eval["boundary_dice"],
    marker="o",
    label="physical-aware",
)
plt.plot(
    standard_eval["step"],
    standard_eval["boundary_dice"],
    marker="o",
    label="ordinary 3x3x3",
)
plt.xlabel("Optimizer step")
plt.ylabel("Boundary Dice")
plt.title("Boundary memorization")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 13. Save histories and final weights

These are diagnostic artifacts. They are stored under `runs/` and should not be committed.


In [ ]:
physical_history_df.to_csv(
    RUN_DIR / "physical_aware_history.csv",
    index=False,
)

standard_history_df.to_csv(
    RUN_DIR / "ordinary_3x3x3_history.csv",
    index=False,
)

comparison.to_csv(
    RUN_DIR / "comparison.csv",
    index=False,
)

torch.save(
    physical_state,
    RUN_DIR / "physical_aware_final.pt",
)

torch.save(
    standard_state,
    RUN_DIR / "ordinary_3x3x3_final.pt",
)

summary = {
    "sample": sample,
    "train_steps": TRAIN_STEPS,
    "eval_every": EVAL_EVERY,
    "physical_aware": physical_final,
    "ordinary_3x3x3": standard_final,
}

with (RUN_DIR / "summary.json").open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
        default=lambda value: (
            value.tolist()
            if isinstance(value, np.ndarray)
            else value
        ),
    )

print("Saved diagnostic run to:")
print(RUN_DIR)


## 14. Reconstruct full-resolution dense predictions for inspection

This runs one evaluation forward per trained model and copies only the three dense probability volumes to CPU.


In [ ]:
@torch.no_grad()
def render_dense_probabilities(
    model_factory,
    state,
):
    gc.collect()
    torch.cuda.empty_cache()

    model = model_factory().to(device)
    model.load_state_dict(
        state["model"],
        strict=True,
    )
    model.eval()

    inputs = _make_device_inputs()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = model(**inputs)

    dense = {
        key.removesuffix("_logits"): (
            torch.sigmoid(value[0, 0].float())
            .cpu()
            .numpy()
            .astype(np.float16)
        )
        for key, value in outputs["dense"].items()
    }

    del outputs, inputs
    model.cpu()
    del model

    gc.collect()
    torch.cuda.empty_cache()

    return dense


physical_dense = render_dense_probabilities(
    lambda: PhysicalAwareSpatialOnly(cfg),
    physical_state,
)

standard_dense = render_dense_probabilities(
    lambda: StandardConvSpatialOnly(cfg),
    standard_state,
)

print("Physical arrays:", physical_dense.keys())
print("Standard arrays:", standard_dense.keys())


## 15. Fast central-slice visualization

This is only a quick check. The optional Napari block below is more useful for 3D inspection.


In [ ]:
raw = (
    spatial_inputs_cpu[0, 0]
    .float()
    .cpu()
    .numpy()
)

gt_labels = (
    torch.as_tensor(targets[0]["label_map"])
    .cpu()
    .numpy()
)

z = raw.shape[0] // 2

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9),
)

axes[0, 0].imshow(raw[z], cmap="gray")
axes[0, 0].set_title(f"Raw z={z}")

axes[0, 1].imshow(gt_labels[z] > 0)
axes[0, 1].set_title("GT foreground")

axes[0, 2].imshow(
    torch.as_tensor(targets[0]["boundary"])[z]
)
axes[0, 2].set_title("GT boundary")

axes[1, 0].imshow(
    physical_dense["foreground"][z],
    vmin=0,
    vmax=1,
)
axes[1, 0].set_title("Physical-aware foreground")

axes[1, 1].imshow(
    standard_dense["foreground"][z],
    vmin=0,
    vmax=1,
)
axes[1, 1].set_title("Ordinary 3x3x3 foreground")

difference = (
    standard_dense["foreground"][z].astype(np.float32)
    - physical_dense["foreground"][z].astype(np.float32)
)
axes[1, 2].imshow(difference)
axes[1, 2].set_title("Standard - physical")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 16. Optional Napari 3D comparison

Set `OPEN_NAPARI = True` only if Napari is available in the current environment.


In [ ]:
OPEN_NAPARI = True

if OPEN_NAPARI:
    import napari

    spacing_zyx_um = tuple(
        float(v)
        for v in spacing_um_cpu[0]
    )

    current_labels = (
        batch["instance_labels"][0]
        .cpu()
        .numpy()
        .astype(np.int32, copy=False)
    )

    viewer = napari.Viewer(ndisplay=3)

    viewer.add_image(
        raw,
        name="Raw",
        scale=spacing_zyx_um,
    )

    viewer.add_labels(
        gt_labels.astype(np.int32, copy=False),
        name="GT instances",
        scale=spacing_zyx_um,
    )

    viewer.add_labels(
        current_labels,
        name="Current CC instances",
        scale=spacing_zyx_um,
        visible=False,
    )

    viewer.add_image(
        physical_dense["foreground"],
        name="Physical-aware foreground",
        scale=spacing_zyx_um,
        contrast_limits=(0, 1),
        opacity=0.7,
    )

    viewer.add_image(
        standard_dense["foreground"],
        name="Ordinary 3x3x3 foreground",
        scale=spacing_zyx_um,
        contrast_limits=(0, 1),
        opacity=0.7,
        visible=False,
    )

    viewer.add_image(
        physical_dense["boundary"],
        name="Physical-aware boundary",
        scale=spacing_zyx_um,
        contrast_limits=(0, 1),
        visible=False,
    )

    viewer.add_image(
        standard_dense["boundary"],
        name="Ordinary 3x3x3 boundary",
        scale=spacing_zyx_um,
        contrast_limits=(0, 1),
        visible=False,
    )

    viewer.add_image(
        physical_dense["center_heatmap"],
        name="Physical-aware center heatmap",
        scale=spacing_zyx_um,
        contrast_limits=(0, 1),
        visible=False,
    )

    viewer.add_image(
        standard_dense["center_heatmap"],
        name="Ordinary 3x3x3 center heatmap",
        scale=spacing_zyx_um,
        contrast_limits=(0, 1),
        visible=False,
    )

    napari.run()
else:
    print("Napari block skipped. Set OPEN_NAPARI=True to inspect in 3D.")


## 17. Interpretation guide

After the default 30 steps:

### Case A — current physical-aware backbone learns strongly

Example pattern:

```text
physical-aware foreground Dice  → high
physical-aware inside prob      > outside prob
physical-aware boundary Dice    rises clearly
```

Then the unusual axis-factorized convolution is **capable of memorizing this scene**. The full STIR-Net failure is more likely due to query/matching/native-mask interactions.

### Case B — ordinary 3×3×3 learns, physical-aware does not

This is the strongest evidence that the current physical-aware spatial operator / reduced cross-axis capacity is a core problem.

Do not immediately replace it permanently; first inspect feature behavior and test a better physical-space operator.

### Case C — both fail

Then convolution family alone is not the explanation. Inspect:

- dense target construction;
- dense loss balance;
- learning rate / optimization;
- reduced backbone capacity;
- gradient flow.

### Case D — both improve but remain modest

Use the continuation block below to extend the **same initialized runs** toward 100 steps before drawing a conclusion.


## 18. Optional continuation to 100 total steps

This block is disabled by default. It resumes both models from their saved optimizer/scaler/model state.

Set:

```python
RUN_EXTENSION = True
EXTRA_STEPS = 70
```

to continue the default 30-step runs to 100 total steps.


In [ ]:
RUN_EXTENSION = False
EXTRA_STEPS = 70

if RUN_EXTENSION:
    physical_state, physical_history, physical_final = train_spatial_model(
        lambda: PhysicalAwareSpatialOnly(cfg),
        name="A — physical-aware continuation",
        steps=EXTRA_STEPS,
        initial_state=physical_state,
        initial_history=physical_history,
    )

    standard_state, standard_history, standard_final = train_spatial_model(
        lambda: StandardConvSpatialOnly(cfg),
        name="B — ordinary 3x3x3 continuation",
        steps=EXTRA_STEPS,
        initial_state=standard_state,
        initial_history=standard_history,
    )

    physical_history_df = pd.DataFrame(physical_history)
    standard_history_df = pd.DataFrame(standard_history)

    comparison = pd.DataFrame(
        [
            {
                "model": "physical_aware",
                **physical_final,
            },
            {
                "model": "ordinary_3x3x3",
                **standard_final,
            },
        ]
    )

    physical_history_df.to_csv(
        RUN_DIR / "physical_aware_history.csv",
        index=False,
    )
    standard_history_df.to_csv(
        RUN_DIR / "ordinary_3x3x3_history.csv",
        index=False,
    )
    comparison.to_csv(
        RUN_DIR / "comparison.csv",
        index=False,
    )
    torch.save(
        physical_state,
        RUN_DIR / "physical_aware_final.pt",
    )
    torch.save(
        standard_state,
        RUN_DIR / "ordinary_3x3x3_final.pt",
    )

    print("Continuation complete.")
    display(comparison)
else:
    print("Extension disabled.")
